In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

In [18]:
df = pd.read_parquet('archive/dataset_vigna.parquet')
df.head(5)

,id,sensor_id,timestamp,temperature,humidity,moisture,grape_count,health_status,estimated_liters,leaf_healthy_count,leaf_stress_count,leaf_disease_count,sector_id,external_id,weather_temp,weather_humidity,weather_rain
0,1617,272,2026-04-21 13:00:00,27.50,49.02,43.83,15.0,Healthy,0.20,96,1,0,1,S-01,18.18,54.403301,0.9
1,1618,273,2026-04-21 13:00:00,28.71,47.63,25.60,20.0,Healthy,0.41,87,1,0,2,S-02,18.18,54.403301,0.9
2,1620,275,2026-04-21 13:00:00,27.62,50.34,26.99,30.0,Healthy,0.59,81,0,0,4,S-04,18.18,54.403301,0.9
3,1621,276,2026-04-21 13:00:00,28.67,46.48,30.77,23.0,Healthy,0.21,87,3,0,5,S-05,18.18,54.403301,0.9
4,1622,277,2026-04-21 13:00:00,28.19,49.80,36.16,19.0,Healthy,0.27,93,4,1,6,S-06,18.18,54.403301,0.9


In [19]:
df.dtypes

id                      int64
sensor_id               int64
timestamp                 str
temperature           float64
humidity              float64
moisture              float64
grape_count           float64
health_status             str
estimated_liters      float64
leaf_healthy_count      int64
leaf_stress_count       int64
leaf_disease_count      int64
sector_id               int64
external_id               str
weather_temp          float32
weather_humidity      float32
weather_rain          float32
dtype: object

In [20]:
# 1. Assicurati che il timestamp sia in formato data
df['timestamp'] = pd.to_datetime(df['timestamp'])
# 2. CREA LA COLONNA HOUR (quella che manca!)
df['hour'] = df['timestamp'].dt.hour
df = df.sort_values('timestamp')

In [21]:
# Creiamo la colonna target: l'umidità che ci sarà tra 24 record (se i tuoi dati sono orari, sono 24 ore)
df['target_moisture'] = df['moisture'].shift(-24)

# Eliminiamo le ultime 24 righe perché non hanno un "domani" da confrontare
df_prep = df.dropna()

print("Dati pronti per l'addestramento!")


Dati pronti per l'addestramento!


In [23]:
# 1. Selezioniamo le variabili di input (X) e quella da indovinare (y)
features = ['moisture', 'temperature', 'weather_temp', 'hour', 'sector_id']
X = df_prep[features]
y = df_prep['target_moisture']

# 2. Dividiamo in Training set (studio) e Test set (esame)
# Usiamo shuffle=False perché i dati sono una serie temporale (il passato spiega il futuro)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

# 3. Creiamo e addestriamo il modello
modello = RandomForestRegressor(n_estimators=100, random_state=42)
modello.fit(X_train, y_train)

# 4. Facciamo le previsioni sui dati che il modello non ha mai visto
previsioni = modello.predict(X_test)

# 5. Calcoliamo l'errore medio
errore = mean_absolute_error(y_test, previsioni)

print(f"Modello addestrato!")
print(f"Errore medio nelle previsioni: {errore:.2f}")


Modello addestrato!
Errore medio nelle previsioni: 5.90


In [27]:
# 1. Definiamo cosa significa "Malata"
# Creiamo una colonna 1 se c'è malattia, 0 se è sana o solo stressata
df['is_diseased'] = df['health_status'].apply(lambda x: 1 if x == 'Disease Detected' else 0)

# 2. Guardiamo al futuro (tra 3 giorni = 72 ore)
df['target_disease'] = df['is_diseased'].shift(-72)

# 3. Puliamo i dati vuoti
df_disease = df.dropna(subset=['target_disease'])

print("Dati pronti per la classificazione!")
print(f"Casi di malattia trovati nel dataset: {df_disease['is_diseased'].sum()}")


Dati pronti per la classificazione!
Casi di malattia trovati nel dataset: 22


In [29]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# Features: aggiungiamo anche l'umidità delle foglie (leaf_healthy_count) che è importante
features_disease = ['moisture', 'temperature', 'weather_humidity', 'weather_rain', 'leaf_healthy_count', 'hour']
X_d = df_disease[features_disease]
y_d = df_disease['target_disease']

# Divisione
X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(X_d, y_d, test_size=0.2, shuffle=False)

# Modello
clf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
clf.fit(X_train_d, y_train_d)

# Esame
predizioni_d = clf.predict(X_test_d)

print("--- REPORT DI CLASSIFICAZIONE ---")
print(classification_report(y_test_d, predizioni_d))


--- REPORT DI CLASSIFICAZIONE ---
              precision    recall  f1-score   support

         0.0       0.92      1.00      0.96        59
         1.0       0.00      0.00      0.00         5

    accuracy                           0.92        64
   macro avg       0.46      0.50      0.48        64
weighted avg       0.85      0.92      0.88        64



/Users/lorenzodimaio/Documents/Iot_project/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/lorenzodimaio/Documents/Iot_project/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/lorenzodimaio/Documents/Iot_project/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.


In [33]:
SOGLIA_GELO = 25  # Impostala a 15 o 20 se i tuoi dati di test sono tutti caldi

# Target: 1 se la temperatura tra 6 ore sarà sotto la soglia, altrimenti 0
df['is_frost_coming'] = (df['temperature'].shift(-6) < SOGLIA_GELO).astype(int)

# Curiosità: creiamo una feature "Trend": la temperatura sta scendendo o salendo?
df['temp_trend'] = df['temperature'] - df['temperature'].shift(1)

df_frost = df.dropna(subset=['is_frost_coming', 'temp_trend'])

print(f"Casi di gelo previsti nel dataset: {df_frost['is_frost_coming'].sum()}")


Casi di gelo previsti nel dataset: 278


In [34]:
# Features: La temperatura attuale e il trend sono i più importanti qui
features_frost = ['temperature', 'temp_trend', 'weather_temp', 'hour', 'weather_humidity']
X_f = df_frost[features_frost]
y_f = df_frost['is_frost_coming']

# Split
X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(X_f, y_f, test_size=0.2, shuffle=False)

# Usiamo sempre class_weight='balanced' perché il gelo è un evento raro
clf_frost = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
clf_frost.fit(X_train_f, y_train_f)

# Previsioni
pred_frost = clf_frost.predict(X_test_f)

print("--- PERFORMANCE ALLERTA GELO ---")
print(classification_report(y_test_f, pred_frost))


--- PERFORMANCE ALLERTA GELO ---
              precision    recall  f1-score   support

           0       0.88      1.00      0.93        28
           1       1.00      0.92      0.96        51

    accuracy                           0.95        79
   macro avg       0.94      0.96      0.95        79
weighted avg       0.96      0.95      0.95        79

